# Freight rate assessment — Stage 2: Validation and baselines

Run the cells from top to bottom. This notebook is a self-contained notebook version of `stage2_validation.py`; it reads `train-test.csv` beside the notebook and writes metrics under `outputs/stage2/`.

In VS Code, select the project's Python interpreter/kernel (`.\\env\\Scripts\\python.exe`). If needed, install dependencies from PowerShell in the project folder:

```powershell
.\env\Scripts\python.exe -m pip install -r requirements.txt
```

The holdout is the final two months of labeled data. Four expanding-window folds evaluate May through August. Random splits are avoided because mixing earlier and later loads between training and evaluation leaks future-period information and does not represent the intended future prediction task.

## 1. Imports, paths, and split configuration

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
TRAIN_PATH = ROOT / "train-test.csv"
OUTPUT_DIR = ROOT / "outputs" / "stage2"
HOLDOUT_START = pd.Timestamp("2025-09-01")
CV_VALIDATION_MONTHS = pd.date_range("2025-05-01", "2025-08-01", freq="MS")
TARGET_CAP_QUANTILE = 0.99
RIDGE_ALPHA = 10.0

NUMERIC_FEATURES = [
    "log_distance",
    "weight",
    "market_index",
    "quote_signal",
    "days_since_start",
    "day_of_year_sin",
    "day_of_year_cos",
]
CATEGORICAL_FEATURES = ["pickup", "delivery", "equipment"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print("Input:", TRAIN_PATH)
print("Output folder:", OUTPUT_DIR)

Input: c:\Users\redni\Downloads\ML-Assessment\train-test.csv
Output folder: c:\Users\redni\Downloads\ML-Assessment\outputs\stage2


## 2. Load data and prepare features

Weight is converted to an absolute value. Numeric and categorical imputers are inside the modeling pipeline, so each imputer is fitted only on its training partition. The time features are created deterministically.

In [2]:
def load_data():
    data = pd.read_csv(TRAIN_PATH)
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    if data["date"].isna().any():
        raise ValueError("Training data contains invalid dates.")
    if data["posted_rate"].isna().any() or (data["posted_rate"] <= 0).any():
        raise ValueError("Training target must contain only positive, non-missing rates.")
    if (data["distance"] <= 0).any():
        raise ValueError("Distance must be positive for the rate-per-mile baseline.")
    return data.sort_values("date").reset_index(drop=True)


def make_features(frame, date_origin):
    features = frame.copy()
    features["weight"] = pd.to_numeric(features["weight"], errors="coerce").abs()
    features["log_distance"] = np.log1p(features["distance"].astype(float))
    features["days_since_start"] = (
        features["date"] - date_origin
    ).dt.total_seconds() / 86400
    day_angle = 2 * np.pi * features["date"].dt.dayofyear / 365.25
    features["day_of_year_sin"] = np.sin(day_angle)
    features["day_of_year_cos"] = np.cos(day_angle)
    return features[FEATURES]


data = load_data()
print("Shape:", data.shape)
print("Date range:", data["date"].min().date(), "to", data["date"].max().date())
print("Missing weight:", data["weight"].isna().sum())
print("Negative weights converted to absolute values:", (data["weight"] < 0).sum())

Shape: (48000, 14)
Date range: 2025-01-01 to 2025-10-31
Missing weight: 300
Negative weights converted to absolute values: 292


## 3. Baselines, fold-fitted preprocessing, and metrics

The first baseline predicts the training-split median rate per mile multiplied by distance. Ridge is fitted to a log target capped at the training split's 99th percentile; evaluation rates remain uncapped.

In [3]:
def build_ridge_pipeline():
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    preprocessing = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ]
    )
    return Pipeline(
        steps=[
            ("preprocessing", preprocessing),
            ("ridge", Ridge(alpha=RIDGE_ALPHA)),
        ]
    )


def calculate_metrics(actual, predicted):
    predicted = np.maximum(predicted, 0)
    residual = predicted - actual
    return {
        "MAE": float(np.mean(np.abs(residual))),
        "RMSE": float(np.sqrt(np.mean(residual**2))),
        "MAPE_pct": float(np.mean(np.abs(residual / actual)) * 100),
        "RMSLE": float(
            np.sqrt(np.mean((np.log1p(predicted) - np.log1p(actual)) ** 2))
        ),
    }


def fit_and_score(train, evaluation, date_origin, split_name):
    if train.empty or evaluation.empty:
        raise ValueError(f"{split_name} has an empty train or evaluation partition.")
    if train["date"].max() >= evaluation["date"].min():
        raise ValueError(f"{split_name} is not strictly time-ordered.")

    y_train = train["posted_rate"].to_numpy(dtype=float)
    y_eval = evaluation["posted_rate"].to_numpy(dtype=float)
    x_train = make_features(train, date_origin)
    x_eval = make_features(evaluation, date_origin)

    median_rate_per_mile = float(
        (train["posted_rate"] / train["distance"]).median()
    )
    baseline_prediction = median_rate_per_mile * evaluation["distance"].to_numpy()

    target_cap = float(train["posted_rate"].quantile(TARGET_CAP_QUANTILE))
    capped_log_target = np.log1p(np.minimum(y_train, target_cap))
    model = build_ridge_pipeline()
    model.fit(x_train, capped_log_target)
    ridge_prediction = np.maximum(np.expm1(model.predict(x_eval)), 0)

    records = []
    for model_name, prediction in (
        ("median_rate_per_mile", baseline_prediction),
        ("ridge_log_target", ridge_prediction),
    ):
        records.append(
            {
                "split": split_name,
                "train_start": train["date"].min().date().isoformat(),
                "train_end": train["date"].max().date().isoformat(),
                "evaluation_start": evaluation["date"].min().date().isoformat(),
                "evaluation_end": evaluation["date"].max().date().isoformat(),
                "n_train": len(train),
                "n_evaluation": len(evaluation),
                "model": model_name,
                "ridge_target_cap_train_p99": target_cap,
                **calculate_metrics(y_eval, prediction),
            }
        )
    return records

## 4. Time-based holdout and expanding-window CV

The final two labeled months (September–October) are held out. CV folds train on all dates before each validation month and score the next month.

In [4]:
date_origin = data["date"].min()

holdout_train = data.loc[data["date"] < HOLDOUT_START]
holdout_eval = data.loc[data["date"] >= HOLDOUT_START]
holdout_records = fit_and_score(
    holdout_train, holdout_eval, date_origin, "holdout_sep_oct"
)

cv_records = []
for validation_month in CV_VALIDATION_MONTHS:
    next_month = validation_month + pd.offsets.MonthBegin(1)
    fold_train = data.loc[data["date"] < validation_month]
    fold_eval = data.loc[
        (data["date"] >= validation_month) & (data["date"] < next_month)
    ]
    fold_name = f"cv_{validation_month:%Y_%m}"
    cv_records.extend(fit_and_score(fold_train, fold_eval, date_origin, fold_name))

holdout = pd.DataFrame(holdout_records)
cv = pd.DataFrame(cv_records)

print("Holdout partition:", len(holdout_train), "train rows and", len(holdout_eval), "evaluation rows")
print("CV folds:", ", ".join(CV_VALIDATION_MONTHS.strftime("%Y-%m")))

Holdout partition: 38477 train rows and 9523 evaluation rows
CV folds: 2025-05, 2025-06, 2025-07, 2025-08


## 5. Results and saved metrics

In [5]:
metric_columns = ["model", "n_train", "n_evaluation", "MAE", "RMSE", "MAPE_pct", "RMSLE"]
print("Time-based holdout metrics (September–October 2025):")
display(holdout[metric_columns].round(4))

print("Expanding-window CV metrics by fold:")
display(cv[["split", *metric_columns]].round(4))

print("Expanding-window CV mean by model:")
display(cv.groupby("model")[["MAE", "RMSE", "MAPE_pct", "RMSLE"]].mean().round(4))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
holdout_path = OUTPUT_DIR / "holdout_metrics.csv"
cv_path = OUTPUT_DIR / "cv_metrics.csv"
holdout.to_csv(holdout_path, index=False)
cv.to_csv(cv_path, index=False)
print("Saved:", holdout_path.relative_to(ROOT))
print("Saved:", cv_path.relative_to(ROOT))

Time-based holdout metrics (September–October 2025):


,model,n_train,n_evaluation,MAE,RMSE,MAPE_pct,RMSLE
0,median_rate_per_mile,38477,9523,256.9529,684.2479,11.6315,0.1981
1,ridge_log_target,38477,9523,161.8469,651.8776,6.6795,0.1656


Expanding-window CV metrics by fold:


,split,model,n_train,n_evaluation,MAE,RMSE,MAPE_pct,RMSLE
0,cv_2025_05,median_rate_per_mile,19110,4913,224.2299,605.0955,11.5414,0.2002
1,cv_2025_05,ridge_log_target,19110,4913,131.9399,566.9741,6.0003,0.1593
2,cv_2025_06,median_rate_per_mile,24023,4783,257.3009,732.6369,11.9582,0.2054
3,cv_2025_06,ridge_log_target,24023,4783,148.6917,715.7491,5.0138,0.1525
4,cv_2025_07,median_rate_per_mile,28806,4912,241.6473,678.1671,10.9393,0.1883
5,cv_2025_07,ridge_log_target,28806,4912,159.2543,629.5166,6.8251,0.1540
6,cv_2025_08,median_rate_per_mile,33718,4759,260.5235,673.9639,11.4585,0.1909
7,cv_2025_08,ridge_log_target,33718,4759,95.1402,614.4405,4.4550,0.1546


Expanding-window CV mean by model:


,MAE,RMSE,MAPE_pct,RMSLE
model,,,,
median_rate_per_mile,245.9254,672.4659,11.4744,0.1962
ridge_log_target,133.7565,631.6701,5.5736,0.1551


Saved: outputs\stage2\holdout_metrics.csv
Saved: outputs\stage2\cv_metrics.csv
